In [1]:
import csv
import os

def load_and_clean(filepath: str) -> list:
    """
    Load sales data from a CSV file and clean/convert types.

    Returns a list of dicts with correct types and a 'revenue' field.
    Skips rows with missing or invalid data.
    """
    if not os.path.exists(filepath):
        print(f"Error: file not found — {filepath}")
        return []

    sales    = []
    skipped  = 0

    with open(filepath, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            try:
                qty   = int(row["quantity"])
                price = float(row["unit_price"])
                if qty < 0 or price < 0:
                    skipped += 1
                    continue              # skip invalid rows
                sales.append({
                    "date":       row["date"].strip(),
                    "product":   row["product"].strip(),
                    "quantity":  qty,
                    "unit_price": price,
                    "revenue":   round(qty * price, 2),
                })
            except (ValueError, KeyError):
                skipped += 1             # skip malformed rows

    print(f"Loaded {len(sales)} rows. Skipped {skipped} invalid rows.")
    return sales

In [2]:

def analyse(sales: list) -> dict:
    """
    Compute summary metrics from cleaned sales data.

    Returns a dict containing:
      - total_revenue: float
      - total_units: int
      - by_product: dict mapping product → {revenue, units, avg_price}
      - best_product: str
      - worst_product: str
      - daily_revenue: dict mapping date → revenue
    """
    if not sales:
        return {}

    # Aggregate by product
    by_product = {}
    for row in sales:
        p = row["product"]
        if p not in by_product:
            by_product[p] = {"revenue": 0, "units": 0, "rows": 0}
        by_product[p]["revenue"] += row["revenue"]
        by_product[p]["units"]   += row["quantity"]
        by_product[p]["rows"]    += 1

    # Add average price per product
    for p, data in by_product.items():
        data["avg_price"] = round(data["revenue"] / data["units"], 2)
        data["revenue"]   = round(data["revenue"], 2)

    # Aggregate by date
    daily_revenue = {}
    for row in sales:
        d = row["date"]
        daily_revenue[d] = daily_revenue.get(d, 0) + row["revenue"]

    # Best and worst performing products by revenue
    best  = max(by_product, key=lambda p: by_product[p]["revenue"])
    worst = min(by_product, key=lambda p: by_product[p]["revenue"])

    return {
        "total_revenue": round(sum(r["revenue"] for r in sales), 2),
        "total_units":   sum(r["quantity"] for r in sales),
        "by_product":   by_product,
        "best_product":  best,
        "worst_product": worst,
        "daily_revenue": daily_revenue,
    }

In [3]:

def print_report(results: dict) -> None:
    """Print a formatted summary report to the terminal."""
    if not results:
        print("No data to report.")
        return

    print("\n" + "=" * 50)
    print("  WEEKLY SALES ANALYSIS REPORT")
    print("=" * 50)

    print(f"\n  Total revenue : £{results['total_revenue']:>10,.2f}")
    print(f"  Total units   : {results['total_units']:>10,}")
    print(f"  Best product  : {results['best_product']}")
    print(f"  Worst product : {results['worst_product']}")

    print("\n  --- Revenue by product ---")
    print(f"  {'Product':12} {'Revenue':>10} {'Units':>7} {'Avg price':>10}")
    print("  " + "-" * 44)

    sorted_products = sorted(
        results["by_product"].items(),
        key=lambda x: x[1]["revenue"],
        reverse=True
    )
    for name, data in sorted_products:
        print(
            f"  {name:12} "
            f"£{data['revenue']:>9,.2f} "
            f"{data['units']:>7,} "
            f"£{data['avg_price']:>9.2f}"
        )

    print("\n  --- Revenue by day ---")
    for date, rev in sorted(results["daily_revenue"].items()):
        bar = "█" * int(rev / 500)     # simple bar chart in terminal
        print(f"  {date} £{rev:>8,.2f} {bar}")

    print("\n" + "=" * 50)

In [4]:
def export_summary(results: dict, output_path: str) -> None:
    """Write the product summary to a CSV file."""
    if not results:
        return

    fieldnames = ["product", "revenue", "units", "avg_price"]

    with open(output_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for name, data in sorted(
            results["by_product"].items(),
            key=lambda x: x[1]["revenue"],
            reverse=True
        ):
            writer.writerow({
                "product":   name,
                "revenue":   data["revenue"],
                "units":     data["units"],
                "avg_price": data["avg_price"],
            })

    print(f"\nSummary exported to: {output_path}")


def main():
    """Run the full sales analysis pipeline."""
    sales   = load_and_clean("sales_data.csv")
    results = analyse(sales)
    print_report(results)
    export_summary(results, "sales_summary.csv")


# Run it!
main()

Loaded 10 rows. Skipped 0 invalid rows.

  WEEKLY SALES ANALYSIS REPORT

  Total revenue : £ 18,915.52
  Total units   :      1,843
  Best product  : Widget A
  Worst product : Gadget X

  --- Revenue by product ---
  Product         Revenue   Units  Avg price
  --------------------------------------------
  Widget A     £ 6,415.72     428 £    14.99
  Connector    £ 6,237.50   1,250 £     4.99
  Widget B     £ 3,262.50     145 £    22.50
  Gadget X     £ 2,999.80      20 £   149.99

  --- Revenue by day ---
  2024-01-01 £2,811.30 █████
  2024-01-02 £3,268.90 ██████
  2024-01-03 £2,707.42 █████
  2024-01-04 £5,642.90 ███████████
  2024-01-05 £4,485.00 ████████


Summary exported to: sales_summary.csv
